# Phase 2 Analysis: Hybrid Intraday Execution vs. Phase 1 Baseline

This notebook justifies and analyses the transition from the **Phase 1 Baseline** (100% Imbalance Settlement) to the **Phase 2 Hybrid Intraday Execution** strategy. It provides a side-by-side performance comparison using equity curves, a tear sheet, and an execution breakdown of the active volume slice.

---
## Background: The Problem with 100% Imbalance Settlement (Phase 1)

In Phase 1, every DA position entered at auction was held passively until the settlement window, where the P&L was determined entirely by the spread between the Day-Ahead price and the live Imbalance price (SSP for longs, SBP for shorts). While this approach captured the core directional alpha, it carried a structural weakness: **unhedged cash-out volatility**.

The GB Imbalance mechanism is a balancing-of-last-resort instrument. On days where the system flips direction between the DA auction and the settlement window — a common occurrence driven by late wind revisions, interconnector outages, or demand surprises — the realised imbalance price can diverge sharply from the model's expected exit price. Phase 1 had no mechanism to protect captured gains or cut losses when the intraday market moved against the position before settlement.

---
## Phase 2: Hybrid Intraday Execution

Phase 2 addresses this by splitting each position into two slices at the time of auction entry:

| Slice | Size | Exit Mechanism | Rationale |
|---|---|---|---|
| **Passive** | `baseline_hedge_ratio` (50%) | Always exits at the intraday mid-market | Locks in a guaranteed, liquid exit — removes the full position's dependence on imbalance cash-out |
| **Active** | Remaining 50% | Conditional: TP / SL trigger → intraday mid; else → imbalance settlement | Preserves upside for high-conviction signals while capping tail risk |

The active slice's exit logic:
- **Take Profit**: if `intraday_mid ≥ DA + predicted_spread × take_profit_pct` (for longs), close at mid. Locks in captured alpha before settlement noise can erode it.
- **Stop Loss**: if unrealised loss per MWh exceeds `stop_loss_mwh`, close at mid. Prevents runaway drawdowns from adverse intraday moves.
- **Imbalance fallback**: if neither trigger fires, the active slice still settles at imbalance — preserving the original Phase 1 mechanic for periods where the intraday market does not provide a superior exit.

This hybrid structure is designed to reduce volatility on individual trades without sacrificing the directional alpha identified by the ML model.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

REPO_ROOT = Path("..").resolve()
STARTING_CAPITAL = 50_000.0

# Phase 1: best Sharpe champion from the DA Positioning sweep (s2_rf, Sharpe 3.79)
PHASE1_DIR = REPO_ROOT / "artifacts" / "da_positioning" / "s2_rf" / "trading"

# Phase 2: hybrid execution run under the da_imbalance strategy
PHASE2_DIR = REPO_ROOT / "artifacts" / "da_imbalance" / "xgb_wf_v1" / "trading"

PHASE1_LABEL = "Phase 1 — Imbalance Settlement (s2_rf)"
PHASE2_LABEL = "Phase 2 — Hybrid Intraday (xgb_wf_v1)"

In [ ]:
def load_artifacts(artifact_dir: Path):
    """Load pnl.csv and metrics.json from an artifact directory.
    Returns (pnl_df, metrics_dict) or (None, None) if the directory is absent."""
    pnl_path = artifact_dir / "pnl.csv"
    metrics_path = artifact_dir / "metrics.json"
    if not pnl_path.exists() or not metrics_path.exists():
        return None, None
    pnl_df = pd.read_csv(pnl_path, parse_dates=["time"])
    with open(metrics_path) as fh:
        metrics = json.load(fh)
    return pnl_df, metrics


p1_pnl, p1_metrics = load_artifacts(PHASE1_DIR)
p2_pnl, p2_metrics = load_artifacts(PHASE2_DIR)

print(f"Phase 1 data loaded : {p1_pnl is not None}")
print(f"Phase 2 data loaded : {p2_pnl is not None}")
if p2_pnl is None:
    print(f"  -> Phase 2 artifacts not yet generated. Run the pipeline with config.yaml to produce {PHASE2_DIR}")

---
## 1. Comparative Equity Curve

The equity curve shows the cumulative account value over the test period for each strategy, starting from the same initial capital. A wider gap between the two curves indicates Phase 2 has either captured more upside or suffered lower drawdowns — the latter being the primary design goal of the hybrid execution.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

if p1_pnl is not None:
    eq1 = STARTING_CAPITAL + p1_pnl["pnl"].cumsum()
    ax.plot(p1_pnl["time"], eq1, label=PHASE1_LABEL, color="#1f77b4", linewidth=1.5)

if p2_pnl is not None:
    eq2 = STARTING_CAPITAL + p2_pnl["pnl"].cumsum()
    ax.plot(p2_pnl["time"], eq2, label=PHASE2_LABEL, color="#2ca02c", linewidth=1.5)

ax.axhline(STARTING_CAPITAL, color="grey", linestyle="--", linewidth=0.8, label="Starting Capital (£50k)")
ax.set_title("Comparative Equity Curve: Phase 1 vs Phase 2", fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Account Equity")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
ax.legend(frameon=False)
plt.tight_layout()

asset_dir = Path("assets")
asset_dir.mkdir(exist_ok=True)
plt.savefig(asset_dir / "phase2_equity_curve.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2. Performance Tear Sheet

The tear sheet summarises the four headline risk/return metrics used to evaluate each strategy phase. Key comparisons to watch:

- **Total PnL**: Absolute return over the test period — did the hybrid layer add or destroy value?
- **Sharpe Ratio**: Risk-adjusted return (annualised, daily resolution). The hybrid exit should reduce daily P&L volatility, pushing Sharpe higher even if raw PnL is similar.
- **Max Drawdown**: The worst peak-to-trough loss in £. The stop-loss mechanism of Phase 2 is designed to reduce this significantly.
- **Win Rate**: Fraction of trades with positive P&L. TP triggers should convert marginal winners and prevent them from flipping to losers at settlement.

In [ ]:
def extract_tearsheet_row(label: str, metrics: dict | None) -> dict:
    if metrics is None:
        return {"Strategy": label, "Total PnL (£)": "—", "Sharpe Ratio": "—", "Max Drawdown (£)": "—", "Win Rate": "—"}
    tp = metrics["trading_performance"]
    return {
        "Strategy": label,
        "Total PnL (£)": f"£{tp['total_pnl']:>10,.0f}",
        "Sharpe Ratio": f"{tp['sharpe_ratio']:.3f}",
        "Max Drawdown (£)": f"£{tp['max_drawdown']:>10,.0f}",
        "Win Rate": f"{tp['win_rate']:.1%}",
    }


rows = [
    extract_tearsheet_row(PHASE1_LABEL, p1_metrics),
    extract_tearsheet_row(PHASE2_LABEL, p2_metrics),
]
tear_df = pd.DataFrame(rows).set_index("Strategy")
tear_df

---
## 3. Execution Breakdown — Active Volume Slice

Phase 2 routes the **active slice** (50% of each position) through a conditional exit gate. The pie chart below shows how that active volume was resolved:

- **Take Profit**: intraday mid crossed the TP level before settlement — gains were locked in early.
- **Stop Loss**: intraday loss per MWh exceeded the threshold — position was cut before further deterioration.
- **Imbalance Settlement**: neither trigger fired — the active slice still settled at the imbalance price (Phase 1 behaviour preserved).

For Phase 1, the entire position settled at imbalance by construction (no intraday routing), shown as a reference baseline.

In [ ]:
def get_execution_counts(metrics: dict | None, n_trades_fallback: int = 0) -> tuple[int, int, int]:
    """Return (tp_count, sl_count, imbalance_count) from metrics.
    Falls back to all-imbalance if execution_breakdown is absent (Phase 1 baseline)."""
    if metrics is None:
        return 0, 0, 0
    tp = metrics["trading_performance"]
    eb = tp.get("execution_breakdown", {})
    tp_count = eb.get("active_tp_triggered", 0)
    sl_count = eb.get("active_sl_triggered", 0)
    imb_count = eb.get("active_rode_to_imbalance", 0)
    # Phase 1 artifacts pre-date the execution_breakdown field:
    # infer all trades settled via imbalance.
    if tp_count == 0 and sl_count == 0 and imb_count == 0:
        imb_count = int(tp.get("n_trades", n_trades_fallback))
    return tp_count, sl_count, imb_count


p1_tp, p1_sl, p1_imb = get_execution_counts(p1_metrics)
p2_tp, p2_sl, p2_imb = get_execution_counts(p2_metrics)

COLORS = ["#2ecc71", "#e74c3c", "#3498db"]
LABELS = ["Take Profit", "Stop Loss", "Imbalance Settlement"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, counts, title in [
    (axes[0], (p1_tp, p1_sl, p1_imb), f"Phase 1\n{PHASE1_LABEL.split('—')[0].strip()}"),
    (axes[1], (p2_tp, p2_sl, p2_imb), f"Phase 2\n{PHASE2_LABEL.split('—')[0].strip()}"),
]:
    non_zero = [(v, l, c) for v, l, c in zip(counts, LABELS, COLORS) if v > 0]
    if not non_zero:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes, fontsize=12)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.axis("off")
        continue
    vals, lbls, clrs = zip(*non_zero)
    wedges, texts, autotexts = ax.pie(
        vals,
        labels=lbls,
        colors=clrs,
        autopct=lambda p: f"{p:.1f}%" if p > 2 else "",
        startangle=90,
        wedgeprops={"linewidth": 1, "edgecolor": "white"},
    )
    for at in autotexts:
        at.set_fontsize(9)
    ax.set_title(title, fontsize=11, fontweight="bold")

fig.suptitle("Execution Breakdown — Active Volume Slice", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(asset_dir / "phase2_execution_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()